In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml
import spacy
from corextopic import corextopic as ct
from dvclive import Live
from matplotlib.figure import Figure
from spacy.tokens import DocBin

from job_post_nlp.utils.interactive import try_inter

try_inter()
from job_post_nlp.prepare import corpus_unpack, register_extensions, load_data,register_extensions, load_texts # noqa: E402
from job_post_nlp.utils.find_project_root import find_project_root  # noqa: E402
from job_post_nlp.evaluate import load_model  # noqa: E402
from job_post_nlp.train import load_corpus_split, load_tdm  # noqa: E402
import helpfuncs as hf

/home/b281467@PROD.SITAD.DK/.conda/envs/jobpostnlp/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# store in one dictionary
data = hf.load_everything()
model = data['model']

In [3]:
model.get_top_docs(topic=0, n_docs=10)

[('4021496', np.float64(0.0)),
 ('4021566', np.float64(0.0)),
 ('4516861', np.float64(0.0)),
 ('4021556', np.float64(0.0)),
 ('4021555', np.float64(0.0)),
 ('4516865', np.float64(0.0)),
 ('4021541', np.float64(0.0)),
 ('4021525', np.float64(0.0)),
 ('4021523', np.float64(0.0)),
 ('4021515', np.float64(0.0))]

In [4]:
hf.print_words_in_doc('2837330', data)

god, arbejde, positiv, overblik, søge, person, fremstille, best
western, hotel, jens, baggesen, kok, dygtig, alsidig, kreativ
selvstændig, glad, værdsætte, anden, mening, spændende, velsmagende, mad
selskab, konferencegæst, omhyggelig, køkkenhygiejne, periodevis, travl, periode, varebestilling
sammensætning, menu, sætter, pris, stabilie, kollegaere, god overblik, arbejde selvstændig


In [5]:
hf.print_words_and_text('2837330', data)

Text for document 2837330:
Best Western Hotel Jens Baggesen søger kok *Du er dygtig, alsidig og kreativ kom med godt
 overblik *Kan arbejde selvstændig *En glad og positiv person, der også værdsætter
 andres mening *Arbejde på mindre hotel *Fremstille spændende og velsmagende mad
 til selskaber og konferencegæster *Er omhyggelig med køkkenhygiejne *Periodevis
 travle perioder *Varebestilling og sammensætning af menuer *Sætter pris på gode
 og stabilie kollegaere
Words in document:
god, arbejde, positiv, overblik, søge, person, fremstille, best
western, hotel, jens, baggesen, kok, dygtig, alsidig, kreativ
selvstændig, glad, værdsætte, anden, mening, spændende, velsmagende, mad
selskab, konferencegæst, omhyggelig, køkkenhygiejne, periodevis, travl, periode, varebestilling
sammensætning, menu, sætter, pris, stabilie, kollegaere, god overblik, arbejde selvstændig


In [6]:
hf.print_random(data)


Vacancy ID: 5945161
Text for document 5945161:
Vi mangler ny/nye kollegaer. Kontakt os endelig for mere info! Vi har travlt og har derfor
 brug for en ny VVS montør til ansættelse hurtigst muligt! Vi har brug for en servicesvend
 til alm. service i ejendomme og hos private, men også til mindre byggesager.
 Hvis det lyder interessant kontakt hurtigst muligt. Lidt om firmaet og stillingen:
 Buur vvs & kloak ApS er en virksomhed beliggende i Kongens Lyngby, men arbejder i
 hele hovedstadsområdet. Vi er et firma, der har alt fra simple serviceopgaver, renovering
 af badeværelse/køkken samt mindre badeværelses projekter. Vi gør altid vores
 yderste for at levere et godt stykke arbejde, samt være serviceminded og vi forventer
 at vores nye kollega vil gøre det samme. Vi tilbyder … • Et job, hvor du bliver en
 del af et lille men stærkt og dynamisk team. • Et rart arbejdsmiljø med gode kollegaer.
 • En fast stilling under gode arbejdsforhold. • Frihed under ansvar. • Egen firmabil.
 • Der li


virksomhed, forvente, levere, god, stykke, arbejde, arbejdsmiljø, spørgsmål
stilling, send, del, inden, dk, hel, ansøgning, ny
holde, lyde, job, beliggende, køkken, gang, travlt, tilbyde
team, samt, cv, kørekort, fast, hos, vægt, arbejdsforhold
ligge, mødestabil, brug, interessere, velkommen, privat, ansvar, serviceminded
kontakt, ringe, kollega, selvstændigt, ansættelse, service, arbejdsopgave, hurtigst
kollegaa, muligt, stærkt, frihed, dynamisk, interessant, yderst, vvs
aps, prioritere, udland, bo, projekt, firma, kilometer, årligt
styr, endelig, mangle, simpel, info, uddannelsesbevis, badeværelse, gyldig
rar, firmabil, ejendom, renovering, byggesag, montør, lyngby, hovedstadsområde
kongen, buur, serviceopgaver, kloak, weekendtur, firmaadress, badeværelses, servicesvend
socialarrangement, stykke arbejde, spørgsmål stilling, send ansøgning, ansøgning samt, samt cv, fast stilling, velkommen ringe
god kollegaa, hurtigst muligt, arbejde selvstændigt, tilbyde job, frihed ansvar, god arbe

In [7]:
ids_sorted = model.word_freq.argsort()
np.array(model.words)[ids_sorted[-10:]]

array(['ansøgning', 'opgave', 'tilbyde', 'samarbejde', 'samt', 'erfaring',
       'stilling', 'god', 'arbejde', 'søge'], dtype='<U100')

In [14]:
hf.print_doc_containing_word('hjemmefra', data)

9532 documents contains the word 'hjemmefra':

Document 1 (ID: 5419880):
Text for document 5419880:
Ildsjæl og uddannet madhåndværker, med passion for god mad, søges til døgninstitution
 Brohuset, Middelfart kommune Pr. 1. oktober 2021 bliver der en stilling som køkkenassistent
 ledig. Stillingen er på 30 timer pr. uge. Arbejdstiden er fra kl. 6.30
 – 12.30 Om os: Brohuset er en døgninstitution (hjem) beliggende i Nr. Åby, under Middelfart
 kommune, hvor der er plads til 14 børn og unge i alderen 6-18 år + 1 akutplads og
 en familielejlighed. Det er Brohusets opgave at stabilisere, samt skabe udvikling
 i barnets eller den unges sociale og psykiske situation, ligesom vi medvirker til
 at stabilisere barnets familie. Vi modtager børn med vidt forskellige problemstillinger,
 på forskellige følelsesmæssige og udviklingsmæssige niveauer, men generelt
 karakteriseres deres symptommønstre ved f.eks. kontaktvanskeligheder,
 følelsesmæssig forvirring, overtilpasning og indtræden i voksenroller

In [11]:
hf.print_doc_containing_word('hjemme', data)

Documents containing the word 'hjemme':

Document 1 (ID: 4604669):
Text for document 4604669:
Er du den nye medarbejder som vi står og skal bruge?Vi søger 1 ny medarbejder som *(afløse
 nu) eller fast chauffør efter aftalemed hjemzone i hvidovre(København), til kørsel
 i en stor bil med trappemaskine, inden for MOVIA/FLEX trafik området, med kørsel
 af borger til og fra sygehuse, læger, privat hjem m.v., Du skal kunne tale og forstå
 dansk, være velsordineret, have en godt gemyt, og kunne lide at have med personlig
 service at gøre.Arbejdstiden kan ligge mellem 06:00 og 20:00, fordelt over alle ugens
 dage, nærmere aftale ved ansættelsen. Der vil være tale om en fast stilling, hvori
 du skal påregne i indgå i vagtordninger for delt ud over 4 til 5 ugedage. Du vil blive
 provisions aflønnet efter overenskomst. Bopæl i Storkøbenhavn / Nordsjælland
 vil være en fordel da der er mulighed for at have bussen med hjemme. Du bedes sende din
 ansøgning til flextrafik15@gmail.com For at komme i 